# Домашняя работа 9. PCA изнутри и выбор числа компонент

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| К семинару | занятие 9 — Метод главных компонент, SVD и многомерное шкалирование |
| Опора | материал семинара 9 и лекций до него |
| Ожидаемое время | 3–4 часа |

Последняя домашняя работа курса. Реализуем PCA двумя путями — через собственные векторы ковариационной матрицы и через SVD — и увидим, почему на практике всегда выбирают второй. Затем разберёмся, что правило «95 % дисперсии» никак не связано с качеством решения задачи.

## Что нужно сдать

Заполненный ноутбук, в котором:

1. выполнены все ячейки с `# TODO`, код исполняется сверху вниз без ошибок
   в свежем ядре (Kernel → Restart & Run All);
2. под каждым заданием заполнена ячейка **Вывод** — своими словами;
3. графики подписаны: заголовок, оси, легенда;
4. вариант ваш собственный (ячейка ниже).

> Списывание видно сразу: у каждого студента свой датасет и свой набор методов.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from labdata import load_personal

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=9)
describe_variant(variant)

---
# Задача 1. PCA двумя способами

**Путь 1 (по определению).** Построить $C = \frac1\ell X^{\mathsf T}X$ и найти
её собственные пары: $\lambda_j$ — дисперсии вдоль компонент, $w_j$ — сами
компоненты.

**Путь 2 (через SVD).** По теореме 8.9 $X = U\Sigma V^{\mathsf T}$, откуда
$X^{\mathsf T}X = V\Sigma^2V^{\mathsf T}$: правые сингулярные векторы $V$ — это
и есть главные компоненты, а $\lambda_j = \sigma_j^2/\ell$. Матрица $C$ при
этом вообще не строится.

In [ ]:
class MyPCA:
    """PCA двумя способами: 'eig' -- через собственные векторы C, 'svd' -- через SVD.

    fit(X): центрировать; далее
      'eig': C = Xc^T Xc / l, np.linalg.eigh, отсортировать по УБЫВАНИЮ;
      'svd': np.linalg.svd(Xc), lambda_j = sigma_j^2 / l, компоненты -- строки Vt.
      Зафиксируйте знак каждой компоненты (например, по максимальной по модулю
      координате), иначе результаты не будут воспроизводимы.
      Сохраните mean_, components_, explained_variance_, explained_variance_ratio_.
    transform(X), inverse_transform(Z).
    """
    raise NotImplementedError

### Задание 1.1. Сверка с конспектом и со `sklearn`

In [ ]:
X_ex = np.array([[2.0, 1.0], [-1.0, -2.0], [1.0, 2.0], [-2.0, -1.0]])
# TODO: 1) обоими способами воспроизведите пример 8.7 конспекта
#          (lambda = 4.5 и 0.5, w_1 = (1,1)/sqrt(2), доля 0.9);
#       2) на wine (после StandardScaler) сверьтесь со sklearn.decomposition.PCA.
#          Внимание: sklearn делит на l-1, а конспект -- на l.

### Задание 1.2. Почему на практике всегда SVD

Собственные числа $C$ равны квадратам сингулярных чисел $X$ (делённым на $\ell$),
поэтому $\mathrm{cond}(C) = \mathrm{cond}(X)^2$ — построив $C$ явно, мы
**возводим обусловленность в квадрат** и теряем вдвое больше верных знаков. Ровно
тот же эффект, что с нормальными уравнениями в домашней работе 2.

**Важно:** матрицу надо собирать **уже центрированной**, иначе центрирование
внутри `fit` изменит спектр и сравнивать будет не с чем.

In [ ]:
# TODO: для kappa из [1e2, 1e4, 1e6, 1e8, 1e10]:
#   1) случайная матрица (300, 6), ЦЕНТРИРОВАТЬ, разложить по SVD;
#   2) подменить сингулярные числа на np.logspace(0, -log10(kappa), 6)
#      и собрать матрицу обратно -- теперь истинные lambda известны точно
#      (lambda_j = sigma_j^2 / l);
#   3) сравнить относительную ошибку САМОГО МАЛОГО собственного числа
#      для методов 'eig' и 'svd'.

> **Вывод.** С какого числа обусловленности подход через $C$ теряет точность и почему? Где мы уже встречали этот эффект?
>
> *(ваш ответ здесь)*

---
# Задача 2. Сколько компонент на самом деле нужно

Правило «оставить 95 % дисперсии» удобно, но оно **никак не связано с качеством
решения задачи**: PCA не видит меток и не знает, какое направление полезно для
классификации.

Постройте два графика в одних осях — накопленную долю дисперсии и качество
классификатора по скользящему контролю, обе как функции $k$, — и посмотрите,
совпадают ли их оптимумы.

In [ ]:
data = load_personal(variant)
Xtr, ytr = data["X_train"], data["y_train"]
if data["task"] == "regression":
    ytr = (ytr > np.median(ytr)).astype(int)
ytr = np.asarray(ytr).astype(int)

# TODO: 1) накопленная доля дисперсии по всем компонентам;
#       2) для каждого k -- качество Pipeline(PCA(k) -> LogisticRegression)
#          по 5-кратному стратифицированному контролю (scoring="roc_auc");
#       3) найдите k по правилу 95% дисперсии и k по максимуму качества.

In [ ]:
# TODO: постройте оба графика в одних осях (вторая ось Y -- через ax1.twinx())
#       и отметьте вертикалями k по дисперсии и k по качеству.

> **Вывод.** Совпали ли два оптимума? В каких задачах разумно выбирать $k$ по дисперсии, а в каких — по качеству итоговой модели?
>
> *(ваш ответ здесь)*

## Итоги домашней работы

Кратко ответьте на вопросы:

1. PCA максимизирует дисперсию проекции. Придумайте задачу, где самое полезное направление имеет наименьшую дисперсию.
2. Оглядываясь на весь курс: какие три вещи вы теперь проверите первым делом, получив чужой ноутбук с результатом «AUC 0.95»?

---

Проверьте перед сдачей: Kernel → Restart & Run All проходит без ошибок,
все ячейки **Вывод** заполнены, графики подписаны.